# PlantPal - Week 3 Data Pipeline
## Part A: Start with the Data

**Date:** 10 November 2026. **Scope:** Parts A, B and C: executed document-preparation pilot.

This is a standard Python `.ipynb` notebook for VS Code/Jupyter, also compatible with Colab.
Saved outputs were produced by executing these cells locally, not in Colab. No model/API key is needed.
This notebook performs ingestion, inspection and text preparation. It does not create embeddings or evaluate model answers.

### Run in VS Code
Open the PlantPal repository folder, open this notebook, select a Python kernel, and choose **Run All**.
Run the setup cell again after a kernel restart. The dependency cell installs `pypdf==6.10.0` if needed.
Raw PDFs remain in ignored `data/raw/`; do not upload them or full extracted text to public GitHub.

### Run in Colab if required by the lecturer
Open this same notebook in Colab and select **Runtime > Run all**. It uses
`/content/week3_project/data/raw` and `data/processed` there. Files in `/content` are temporary.
The lab specifies Colab; a local run is not evidence of a connected Colab runtime.

The lab's optional dummy ZIP cell is deliberately omitted because actual PlantPal sources are used.


## A1 - Requirements and data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| FR1 | Documented light, watering principles, media and space guidance | S01 indoor guide; S02 houseplant brochure; relevant S03 sections | Official NParks material fits tropical gardening. Actual extraction and coverage must be inspected. Community plot planning and excluded chemical advice do not automatically become supported tasks. |
| FR2 | Document title, URL and one-based PDF page references | All three PDFs and their source register | Preserve page identity; a file hash identifies downloaded bytes, not accuracy or permission. |
| RAI1 | Verifiable source support and evidence of missing coverage | Original pages compared with extracted text | Layout loss, hidden text and icons can misrepresent evidence. A successful parser does not establish factual accuracy or support every plant-care question. |

These needs implement **A5 Approved Sources** and prepare inputs for **A4 Knowledge Access** in V1.
Later correctness and citation-support evaluation depend on reliable source/page associations.
O1 source inspection progresses here; user-needs research and a fully verified answer corpus remain pending.

**Selected data type:** unstructured PDF text. **Selected preparation pathway for Part B:** PDF/TXT (A).
Images are part of document layout, not a new image-recognition task. Other dataset candidates in
`docs/dataset_references.md` are not admitted by this notebook.


## A2 - Workspace and source access

The official [download directory](https://gardeningsg.nparks.gov.sg/learn-more-about-gardening/downloads/)
lists the three sources. [Terms of use](https://gardeningsg.nparks.gov.sg/terms-of-use/) were reviewed
on 10 November 2026. They describe personal non-commercial use and reserve other rights;
no open redistribution licence has been established. This notebook uses personal working copies
for inspection. Public outputs are limited to code, metadata, short samples and original observations.
This is not permission to redistribute PDFs, publish full extraction, train a model, or deploy a public service.

Use local copies when present. The acquisition cell fetches missing files from their explicit official URLs.
Alternatively download them yourself into `data/raw/` using the exact filenames in the register.


In [1]:
# Cell 1 - Workspace setup: RUN FIRST after every kernel restart.
from pathlib import Path
import sys, os, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path('/content/week3_project')
else:
    # VS Code normally starts in the repository or notebooks directory.
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    BASE_DIR = next((p for p in candidates if (p / 'docs/week2_requirements.md').is_file()), here)

RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
assert RAW_DIR.is_dir() and PROCESSED_DIR.is_dir()
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Python (VS Code/Jupyter compatible)')
print('Python:', platform.python_version())
print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)


Environment: Local Python (VS Code/Jupyter compatible)
Python: 3.12.14
Raw folder: [local PlantPal workspace]\data\raw
Processed folder: [local PlantPal workspace]\data\processed


In [2]:
# Dependency setup: no model/API packages are needed for Part A.
import importlib.metadata
import subprocess
try:
    parser_version = importlib.metadata.version('pypdf')
except importlib.metadata.PackageNotFoundError:
    parser_version = None
if parser_version != '6.10.0':
    if 'pypdf' in sys.modules:
        raise RuntimeError('Restart the kernel before changing an already-imported pypdf version.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf==6.10.0'])
from pypdf import PdfReader
print('Parser: pypdf', importlib.metadata.version('pypdf'))


Parser: pypdf 6.10.0


In [3]:
# Explicit source register. Expected hashes identify the bytes inspected on 10 November 2026.
SOURCES = [{'source_id': 'S01',
  'filename': 'introduction_to_indoor_gardening.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/5be81ffb-fd47-444d-ab81-bf46b869dd7b/introduction_to_indoor_gardening.pdf',
  'title': 'Introduction to Indoor Gardening',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '21d6ae22ac7f4bbebfa38225d1fb942d2210376a119d350146fc15888a573533',
  'sample_pages': [5, 10],
  'access_date': '2026-11-10'},
 {'source_id': 'S02',
  'filename': 'know_10_houseplants.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/43fdd8a1-d385-428f-8cc2-d5b9b36f6a38/know%2010%20house%20plants.pdf',
  'title': 'Know 10 Houseplants',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '2208509f38b4c3400270c06b3c20a7c2ddd1812ce4433a2fe3e4dd0b9bd60120',
  'sample_pages': [1, 2],
  'access_date': '2026-11-10'},
 {'source_id': 'S03',
  'filename': 'introduction_to_community_gardening.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/e9f8a03b-d2aa-4f29-b16a-0678ea9ca5a9/introduction%20to%20community%20gardening.pdf',
  'title': 'Introduction to Community Gardening',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '33a4f05c6a55d110e2bcda45562c928b7edd440d750e214e1183717b30ab0096',
  'sample_pages': [5, 12],
  'access_date': '2026-11-10'}]
for source in SOURCES:
    print(source["source_id"], source["title"], "-> data/raw/" + source["filename"])


S01 Introduction to Indoor Gardening -> data/raw/introduction_to_indoor_gardening.pdf
S02 Know 10 Houseplants -> data/raw/know_10_houseplants.pdf
S03 Introduction to Community Gardening -> data/raw/introduction_to_community_gardening.pdf


In [4]:
# Acquire missing personal working copies. Never overwrite an existing source silently.
import hashlib, urllib.request
from datetime import datetime, timezone

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for source in SOURCES:
    path = RAW_DIR / source['filename']
    if not path.exists():
        request = urllib.request.Request(source['url'], headers={'User-Agent': 'PlantPal educational document inspection'})
        with urllib.request.urlopen(request, timeout=60) as response:
            content = response.read()
        if not content.startswith(b'%PDF'):
            raise ValueError(f"{source['source_id']}: download is not a PDF")
        path.write_bytes(content)
        print(source['source_id'], 'downloaded')
    else:
        print(source['source_id'], 'using existing personal copy')
    if not path.read_bytes().startswith(b'%PDF'):
        raise ValueError(f"{source['source_id']}: invalid PDF signature")
    actual = sha256_file(path)
    if actual != source['expected_sha256']:
        raise ValueError(f"{source['source_id']}: source bytes changed. Review the new version before updating the source register.")
print('Source signatures and fingerprints verified.')


S01 using existing personal copy
S02 using existing personal copy
S03 using existing personal copy
Source signatures and fingerprints verified.


In [5]:
# Cell 3 - Verify inputs recursively and select the intended files explicitly.
files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(files))
for path in files:
    print(path.relative_to(RAW_DIR).as_posix(), 'bytes:', path.stat().st_size)
assert files, 'No inputs: add the selected PDFs to data/raw.'
for source in SOURCES:
    assert (RAW_DIR / source['filename']).is_file(), source['filename']
print('All three selected PlantPal source files are accessible.')


Input count: 3
introduction_to_community_gardening.pdf bytes: 2450552
introduction_to_indoor_gardening.pdf bytes: 3178940
know_10_houseplants.pdf bytes: 1720974
All three selected PlantPal source files are accessible.


## A3 - Inspect actual content, page structure and ingestion limitations

The next cell extracts page text **in memory** and preserves one-based PDF page numbers.
It does not clean, chunk or export the corpus. Empty extraction is flagged for visual checking,
not automatically labelled a broken document or passed through OCR.
Two representative pages per source provide more than the lab's minimum of two source artefacts/subsets.
Short text samples are deliberately limited to nine words per inspected page.
Full extracted text stays in memory and is not written into this public notebook's outputs.


In [6]:
import json
READERS = {}
PAGE_TEXT = {}
INSPECTION = []
for source in SOURCES:
    path = RAW_DIR / source['filename']
    reader = PdfReader(path)
    if reader.is_encrypted:
        raise ValueError(f"{source['source_id']}: encrypted input requires an authorised readable copy")
    if not len(reader.pages):
        raise ValueError(f"{source['source_id']}: PDF has no pages")
    page_text = [page.extract_text() or '' for page in reader.pages]
    if not any(t.strip() for t in page_text):
        raise ValueError(f"{source['source_id']}: no extractable text; inspect visually before choosing a loader")
    READERS[source['source_id']] = reader
    PAGE_TEXT[source['source_id']] = page_text
    metadata = reader.metadata or {}
    record = {
        'source_id': source['source_id'], 'title': source['title'],
        'filename': source['filename'], 'url': source['url'], 'publisher': source['publisher'],
        'source_sha256': sha256_file(path), 'bytes': path.stat().st_size,
        'pdf_pages': len(reader.pages),
        'characters_per_page': [len(t) for t in page_text],
        'empty_text_pages': [i + 1 for i, t in enumerate(page_text) if not t.strip()],
        'pdf_creation_metadata': str(metadata.get('/CreationDate', 'not supplied')),
        'pdf_modification_metadata': str(metadata.get('/ModDate', 'not supplied')),
        'sample_pages': source['sample_pages'],
    }
    INSPECTION.append(record)
    print('\n', source['source_id'], source['title'])
    print('Pages:', record['pdf_pages'], '| characters:', sum(record['characters_per_page']))
    print('Empty-text pages:', record['empty_text_pages'])
    print('Creation metadata (not a verified publication date):', record['pdf_creation_metadata'])
    for page_no in source['sample_pages']:
        text = page_text[page_no - 1]
        excerpt = ' '.join(text.split()[:9])
        print(f'PDF page {page_no}: {len(text)} characters | sample: {excerpt} ...')
print('\nPASS: three readable PDFs inspected; original page identifiers retained.')



 S01 Introduction to Indoor Gardening
Pages: 24 | characters: 13273
Empty-text pages: [2]
Creation metadata (not a verified publication date): D:20201013090212+08'00'
PDF page 5: 638 characters | sample: Indoor Garden Considerations Space Identify a corner for you ...
PDF page 10: 1000 characters | sample: Plant Necessity - Media A good medium for your ...

 S02 Know 10 Houseplants
Pages: 2 | characters: 9439
Empty-text pages: []
Creation metadata (not a verified publication date): D:20230127115509Z
PDF page 1: 2810 characters | sample: For more information, visit our website at www.nparks.gov.sg/cib or ...
PDF page 2: 6629 characters | sample: Scientific Name Monstera obliqua Plant Characteristics The Window-leaf Monstera ...

 S03 Introduction to Community Gardening
Pages: 20 | characters: 7762
Empty-text pages: []
Creation metadata (not a verified publication date): D:20220602102149+08'00'
PDF page 5: 926 characters | sample: Soil Conditions Plants need good soil to grow well. ...


### Visual comparison recorded on 10 November 2026

- **S01:** PDF page 2 is a photograph with no visible text, explaining its zero extracted characters.
  Page 10 extracts an editorial image-placeholder sentence that is not visible in the rendered page.
  This is an actual extraction artefact to flag before using the text as evidence. Page 5 has multiple
  topic blocks; preserve their headings and boundaries in later preparation.
- **S02:** PDF page 2 displays ten plant profiles in a multi-column layout. Plain extraction starts with
  the bottom-right profile rather than the top-left profile, and several common-name headings appear
  together near the end. Text is readable, but sequence alone is unsafe for associating care text with a plant.
- **S03:** Page 5 mixes general growing guidance with community-garden planning. On page 12,
  care icons sit beside plant names but their meanings are not encoded as corresponding plain-text fields.
  Page 11 explains the icons. Page 9 includes advice outside PlantPal's excluded chemical guidance boundary.

Visual review is separate from automated extraction. These findings were obtained by viewing local
renderings of the originals; the copyrighted page images are not embedded in the public notebook.
On reruns, inspect these same pages in the original PDFs and verify whether the findings still apply.


In [7]:
# Reproducible structure checks for the inspected snapshot; no source text is modified.
checks = {
    'S01_page_2_has_no_extractable_text': not PAGE_TEXT['S01'][1].strip(),
    'S01_page_10_contains_editorial_placeholder': 'Can insert any relevant image here?' in PAGE_TEXT['S01'][9],
    'S02_page_2_starts_with_bottom_right_profile': PAGE_TEXT['S02'][1].lstrip().startswith('Scientific Name\nMonstera obliqua'),
    'S03_page_12_has_plant_name_text': 'Lettuce' in PAGE_TEXT['S03'][11],
}
for name, result in checks.items():
    print(name, ':', result)
assert all(checks.values()), 'Inspection snapshot changed: review the PDF and extraction before proceeding.'
print('These checks reproduce specific observations; they do not prove source accuracy or complete coverage.')


S01_page_2_has_no_extractable_text : True
S01_page_10_contains_editorial_placeholder : True
S02_page_2_starts_with_bottom_right_profile : True
S03_page_12_has_plant_name_text : True
These checks reproduce specific observations; they do not prove source accuracy or complete coverage.


## A4 - Data inventory checkpoint and preparation decision

See `docs/week3_data_inventory.md` for the source register, access review, file identities,
requirement mapping and observed limitations. The next cell saves a **metadata-only Part A report**,
not a prepared-data export or the Part B/C pipeline manifest.

**Part B decision:** use the PDF/TXT pathway. Preserve document and page provenance. Compare
layout-aware/heading-aware handling against simple extraction before choosing cleaning or chunking.
Flag hidden editorial text for exclusion with recorded justification. Do not turn care icons into
plant-specific claims without verifying their association. S03 remains a supplementary candidate;
ingestion does not admit all pages or excluded topics into PlantPal's answer scope.

No source content has been cleaned, chunked, embedded, or scored as a model answer in Part A.


In [8]:
REPORT_DIR = BASE_DIR / 'data' / 'inspection'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = {
    'stage': 'week3-part-a-ingestion-and-inspection',
    'execution_environment': 'Google Colab' if IN_COLAB else 'local Python',
    'execution_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'pypdf_version': importlib.metadata.version('pypdf'),
    'raw_storage': 'data/raw/ (ignored by Git)',
    'records': INSPECTION,
    'snapshot_checks': checks,
    'note': 'Metadata only. No cleaned corpus, model evaluation or Part B/C pipeline results.',
}
report_path = REPORT_DIR / 'week3_part_a_inspection.json'
report_path.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert json.loads(report_path.read_text(encoding='utf-8')) == REPORT
print('Saved and read back:', report_path.relative_to(BASE_DIR).as_posix())
print('Records:', len(REPORT['records']), '| raw PDF pages:', sum(r['pdf_pages'] for r in INSPECTION))


Saved and read back: data/inspection/week3_part_a_inspection.json
Records: 3 | raw PDF pages: 46


## Part A status

- A1: three existing Topic 2 requirements mapped to actual project PDFs.
- A2: local workspace initialised and all three inputs verified. Colab compatibility is provided;
  **a connected Colab execution has not been performed**.
- A3: three PDFs and six representative page subsets inspected, with separate visual comparisons.
- A4: inventory and metadata-only evidence prepared.
- Part B follows below; Part C follows below.

**Public sharing:** retain source notices on personal copies. Before saving/pushing a rerun, check
that no full text, source page images, private paths, credentials or personal input have been added
to outputs. The repository's raw/processed ignores do not remove content embedded in notebook outputs.


## Part B - Prepare document data (B1-B4)

We use the lab's PDF/TXT pathway, not the unrelated image/tabular pathways. This is a deliberately
small, verified preparation pilot, **not the complete PlantPal knowledge base**.

| Source/pages | Admission decision | Reason |
|---|---|---|
| S01 pages 5 and 10 | Include text sections | Indoor conditions and growing media support FR1. Remove only the confirmed hidden editorial placeholder on page 10. |
| S02 both pages | Defer | Page 2 mixes profile headings and bodies; assigning them to plants requires further layout verification. Page 1 is not needed for this pilot. |
| S03 page 6 | Include extracted prose only | Visually checked definitions and general soil guidance. The rendered soil-ratio table is absent from extraction: no ratios or table-derived claims are admitted. |
| All other pages | Not selected | Not yet prepared/verified for this pilot; not necessarily invalid. S01 page 2 is a photo; S03 page 9 contains excluded chemical guidance; icon-based care mappings remain deferred. |

Page 6 of S03 was checked visually against the source on 10 November 2026. Its heading extracts last;
we move that verified heading to the start. We preserve the other prose order and wording.
S01 section boundaries are reconstructed using exact, visually checked headings. These are manual,
source-specific rules, not a general layout parser. Source hashes protect against silently using changed files.

**Comparison:** identical admitted text, fixed 500-character chunks versus paragraph-aware packing
with a 500-character target. Both stay within a PDF page. A paragraph longer than the target stays intact:
500 is a soft target, not a model token limit. No overlap is used. We measure splits inside verified sections
and inside words, inspect the local candidates, then select the approach. This is not semantic chunking.


In [9]:
# Cell 4 / 4A: common helpers and fresh lineage registry on every run
import re
PIPELINE_VERSION = 'plantpal-week3-part-b-v1'
def select_source(relative_path):
    p = (RAW_DIR / relative_path).resolve()
    p.relative_to(RAW_DIR.resolve())
    assert p.is_file(), f'Missing source: {relative_path}'
    return p
source_hash = sha256_file
lineage_entries = {}
def register_lineage(source, output, processing):
    source, output = Path(source).resolve(), Path(output).resolve()
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    assert source.is_file() and output.is_file() and output.stat().st_size
    assert isinstance(processing, str) and processing.strip()
    lineage_entries[(str(source), str(output))] = (source, output, processing)
def save_manifest(records, path):
    payload = {'pipeline_version': PIPELINE_VERSION,
               'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'records': records}
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')

# Cell 5A1: source-specific, verified section boundaries; original files stay untouched.
ADMITTED = {'S01': [5, 10], 'S03': [6]}
SECTIONS = {
    ('S01', 5): ['Indoor Garden\nConsiderations', 'Space', 'Light', 'Water', 'Plant Choice'],
    ('S01', 10): ['Plant Necessity -\nMedia', 'Charcoal', 'Coco Chips'],
}
PAGE_DOCS = []
for sid, pages in ADMITTED.items():
    for page in pages:
        raw = PAGE_TEXT[sid][page-1]
        if (sid, page) == ('S01', 10):
            noise = 'Can insert any relevant image here?'
            assert raw.count(noise) == 1
            raw = raw.replace(noise, '')
        if sid == 'S01':
            markers = SECTIONS[(sid, page)]
            # Match complete heading lines, not words inside the prose.
            positions = []
            for marker in markers:
                matches = list(re.finditer(r'(?m)^'+re.escape(marker)+r'\s*$', raw))
                assert len(matches) == 1, (sid, page, marker)
                positions.append(matches[0].start())
            assert positions == sorted(positions) and positions[0] == 0
            blocks = [raw[a:b] for a,b in zip(positions, positions[1:]+[len(raw)])]
            # Keep page heading with first content section, not as an orphan chunk.
            if page == 5:
                blocks = [blocks[0]+blocks[1]] + blocks[2:]
        else:
            assert raw.rstrip().endswith('Soil Mix')
            raw = raw.rstrip()[:-len('Soil Mix')].rstrip()
            starts = [m.start() for m in re.finditer(r'(?m)^(?:Topsoil refers|Compost refers|Different plants|Sand refers)', raw)]
            assert len(starts) == 4 and starts[0] == 0
            blocks = [raw[a:b] for a,b in zip(starts, starts[1:]+[len(raw)])]
            blocks[0] = 'Soil Mix\n'+blocks[0]
        blocks = [re.sub(r'\s+', ' ', b).strip() for b in blocks]
        assert all(blocks)
        PAGE_DOCS.append({'source_id': sid, 'page': page, 'blocks': blocks,
                          'text': '\n\n'.join(blocks)})
print('Admitted pages:', [(d['source_id'], d['page']) for d in PAGE_DOCS])
print('Verified sections:', sum(len(d['blocks']) for d in PAGE_DOCS))
print('S02 deferred; S03 page 6 table content is not represented.')


Admitted pages: [('S01', 5), ('S01', 10), ('S03', 6)]
Verified sections: 11
S02 deferred; S03 page 6 table content is not represented.


In [10]:
# Compare the SAME cleaned text using character spans for auditable boundaries.
TARGET = 500
def fixed_spans(text):
    return [(i, min(i+TARGET, len(text))) for i in range(0,len(text),TARGET)]
def paragraph_spans(text):
    blocks = list(re.finditer(r'[^\n]+', text))
    spans, start, end = [], None, None
    for b in blocks:
        if start is not None and b.end()-start > TARGET:
            spans.append((start, end))
            start = None
        if start is None:
            start = b.start()
        end = b.end()
    if start is not None:
        spans.append((start, end))
    return spans
CANDIDATES = {}
COMPARISON = []
for method, splitter in [('fixed', fixed_spans), ('paragraph-aware', paragraph_spans)]:
    rows = []
    section_splits = word_splits = 0
    for doc in PAGE_DOCS:
        text = doc['text']
        spans = splitter(text)
        sections = [(m.start(), m.end()) for m in re.finditer(r'[^\n]+', text)]
        for a,b in spans[:-1]:
            section_splits += any(x < b < y for x,y in sections)
            word_splits += text[b-1].isalnum() and text[b].isalnum()
        for a,b in spans:
            rows.append({'source_id':doc['source_id'], 'page':doc['page'],
                         'start':a, 'end':b, 'text':text[a:b].strip()})
        assert re.sub(r'\s+','', ''.join(text[a:b] for a,b in spans)) == re.sub(r'\s+','',text)
    CANDIDATES[method] = rows
    stats = {'method':method,'chunks':len(rows), 'section_splits':section_splits,
             'word_splits':word_splits, 'max_characters':max(len(r['text']) for r in rows),
             'over_target_chunks':sum(len(r['text'])>TARGET for r in rows)}
    COMPARISON.append(stats)
    print(stats)
    # Full candidate samples are LOCAL ONLY: inspect these JSONL files in VS Code.
    candidate_path = PROCESSED_DIR / ('candidate_'+method+'.jsonl')
    candidate_path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows), encoding='utf-8')
    print('Local candidate sample:', candidate_path.relative_to(BASE_DIR).as_posix())
    print('First two chunk metadata:', [{k:v for k,v in r.items() if k!='text'} for r in rows[:2]])
CHOSEN_METHOD = 'paragraph-aware'
assert COMPARISON[1]['section_splits'] == 0
assert COMPARISON[1]['word_splits'] == 0
print('Selected:', CHOSEN_METHOD, '| benefit: verified sections stay intact; trade-off: soft length cap.')


{'method': 'fixed', 'chunks': 5, 'section_splits': 2, 'word_splits': 1, 'max_characters': 500, 'over_target_chunks': 0}
Local candidate sample: data/processed/candidate_fixed.jsonl
First two chunk metadata: [{'source_id': 'S01', 'page': 5, 'start': 0, 'end': 500}, {'source_id': 'S01', 'page': 5, 'start': 500, 'end': 634}]
{'method': 'paragraph-aware', 'chunks': 5, 'section_splits': 0, 'word_splits': 0, 'max_characters': 656, 'over_target_chunks': 1}
Local candidate sample: data/processed/candidate_paragraph-aware.jsonl
First two chunk metadata: [{'source_id': 'S01', 'page': 5, 'start': 0, 'end': 466}, {'source_id': 'S01', 'page': 5, 'start': 468, 'end': 634}]
Selected: paragraph-aware | benefit: verified sections stay intact; trade-off: soft length cap.


### B3 - Selection and limitations

The comparison above is measured on the same three pages, not an assumed retrieval score.
Paragraph-aware packing keeps verified sections intact; fixed character cuts can interrupt a section
or a word. Selected: paragraph-aware, target 500 characters, zero overlap, page-local.
Accepted trade-offs: variable chunk lengths (including above 500), manual heading rules, no cross-page
context, and incomplete source coverage. Preserving sections does not prove answer accuracy.

Full readable candidate samples are exported locally for inspection; public notebook previews contain
only metadata. Inspect both candidate JSONL files before expanding this corpus. No text is paraphrased
or repaired using a language model. Missing table/icon content is not inferred.

**Observed local sample review:** fixed chunks separate the plant-choice heading from the rest of its advice and split a word in the media paragraph. The selected chunks retain those sections, keep charcoal/coco-chip descriptions with their headings, and preserve only the extracted S03 prose. Both methods produce five chunks; selected maximum length is 656 characters. The missing soil-ratio table remains excluded.


In [11]:
# Cell 5A2 and B3: export traceable chunks and validate by reading the JSONL back.
by_id = {s['source_id']:s for s in SOURCES}
records = []
for i, chunk in enumerate(CANDIDATES[CHOSEN_METHOD], 1):
    s = by_id[chunk['source_id']]
    records.append(dict(chunk, record_id=f"{chunk['source_id']}-p{chunk['page']:02d}-{i:04d}",
                        source=s['filename'], title=s['title'], url=s['url'],
                        source_sha256=source_hash(select_source(s['filename'])),
                        processing='verified-sections; paragraph-aware; target-char-500; overlap-0; page-local',
                        coverage_note='S03 p6: prose only; extracted text omits soil-ratio table.' if s['source_id']=='S03' else 'Selected page text only.'))
prepared_path = PROCESSED_DIR / 'prepared_text.jsonl'
prepared_path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in records),encoding='utf-8')
assert prepared_path.is_file() and prepared_path.stat().st_size > 0
loaded = [json.loads(line) for line in prepared_path.read_text(encoding='utf-8').splitlines()]
assert loaded == records and len({r['record_id'] for r in loaded}) == len(loaded)
for r in loaded:
    assert r['text'].strip() and r['title'] and r['url'].startswith('https://')
    assert r['source_id'] in ADMITTED and r['page'] in ADMITTED[r['source_id']]
    assert 1 <= r['page'] <= len(READERS[r['source_id']].pages)
    assert r['source_sha256'] == by_id[r['source_id']]['expected_sha256']
    d = next(d for d in PAGE_DOCS if (d['source_id'],d['page'])==(r['source_id'],r['page']))
    assert r['text'] == d['text'][r['start']:r['end']].strip()
    assert 'Can insert any relevant image here?' not in r['text']
print('Prepared JSONL validation: PASS')
print('Records:',len(loaded),'| bytes:',prepared_path.stat().st_size)
print('Unique source/page pairs:',sorted({(r['source_id'],r['page']) for r in loaded}))
print('Checks establish structure and traceability, not factual accuracy or retrieval quality.')

# B4.1: each source contributing to the shared output has its own lineage entry.
for sid in ADMITTED:
    register_lineage(select_source(by_id[sid]['filename']), prepared_path,
                     'verified-sections; paragraph-aware; target-char-500; overlap-0; page-local')


Prepared JSONL validation: PASS
Records: 5 | bytes: 4766
Unique source/page pairs: [('S01', 5), ('S01', 10), ('S03', 6)]
Checks establish structure and traceability, not factual accuracy or retrieval quality.


In [12]:
# Cell 6A / 6B: save common manifest, then verify saved hashes and references.
assert lineage_entries
lineage_records = []
for (source, output, processing) in lineage_entries.values():
    s = next(s for s in SOURCES if s['filename']==source.name)
    lineage_records.append({
        'source_id':s['source_id'], 'source_filename':source.relative_to(RAW_DIR).as_posix(),
        'source_url':s['url'], 'source_title':s['title'], 'source_sha256':source_hash(source),
        'prepared_output':output.relative_to(BASE_DIR).as_posix(), 'prepared_sha256':source_hash(output),
        'processing':processing, 'admitted_pages':ADMITTED[s['source_id']],
        'record_count':sum(r['source_id']==s['source_id'] for r in records),
        'parser_version':importlib.metadata.version('pypdf'),
        'provenance_limitations':'Selected prose pilot; no redistribution licence established; raw and prepared files local only. S02 deferred. S03 p6 table absent. Hashes do not prove truth or permission.',
    })
manifest_path = BASE_DIR / 'data/week3_manifest.json'
save_manifest(lineage_records, manifest_path)
saved = json.loads(manifest_path.read_text(encoding='utf-8'))
assert saved['pipeline_version'] == PIPELINE_VERSION and saved['generated_at_utc']
assert len(saved['records']) == len(lineage_entries) == 2
for r in saved['records']:
    assert source_hash(RAW_DIR/r['source_filename']) == r['source_sha256']
    assert source_hash(BASE_DIR/r['prepared_output']) == r['prepared_sha256']
print('COMMON LINEAGE VALIDATION: PASS | verified relationships:',len(saved['records']))
comparison_path = BASE_DIR/'data/inspection/week3_part_b_comparison.json'
comparison_path.write_text(json.dumps({'pipeline_version':PIPELINE_VERSION,
    'admitted_pages':ADMITTED,'comparison':COMPARISON,'selected_method':CHOSEN_METHOD,
    'validation':'JSONL round trip, unique IDs, page/source references, content spans and hashes passed',
    'limitations':['Pilot only','Manual verified section boundaries','Soft character target',
                   'No model/retrieval evaluation','S02 deferred','S03 p6 table omitted']},indent=2)+'\n',encoding='utf-8')
print('Saved metadata-only comparison:',comparison_path.relative_to(BASE_DIR).as_posix())


COMMON LINEAGE VALIDATION: PASS | verified relationships: 2
Saved metadata-only comparison: data/inspection/week3_part_b_comparison.json


## Part B status

- B1/B2: document pathway selected; justified cleanup and two chunk configurations executed.
- B3: output read back and validated; selection linked to FR1 context, FR2 page citations, RAI1 source support.
- B4: two source-to-output relationships saved and verified, with source/output SHA-256 checksums.
- Prepared text and candidate samples stay in ignored `data/processed/`. Manifest and comparison evidence
  can be shared, but a fresh clone must run the notebook to regenerate its local prepared files.
- S02 is deferred; this pilot does not claim all three PDFs are ready for retrieval.
- Local execution only; the lecturer's Colab runtime checkpoint remains pending; Part C follows below.

**Lab reference:** Topic 3 Lab, PDF/text pathway pp. 16-20, validation/lineage pp. 35-40.
Source URLs and access notes are in the Part A register and `docs/week3_data_inventory.md`.


## Part C - Executable V1 document pipeline

**C1 / Cells 7, 8A and 9:** one reusable batch runner re-reads source files, inspects them,
prepares the same selected content as Part B, validates stored records and records lineage.
The runner does not consume Part B's in-memory extracted text or prepared files.
Part B output is used only as a regression reference after execution.

Configuration remains: S01 pages 5/10 and S03 page 6, verified section boundaries,
paragraph-aware packing, 500-character soft target, zero overlap, no cross-page chunks.
This is ETL into prepared JSONL; keeping an unchanged raw working copy does not change that choice.
The runner is called twice; timestamps may differ, but source/output hashes, sizes and mappings must match.
These are local repeatability checks, not cross-environment reproducibility or answer-quality evaluation.

The pipeline expands Week 2 **A5 (source admission/preparation)**, producing page-tagged data for
future **A4 (knowledge access)**. No retrieval, model API, embedding, training or deployment is added.


In [13]:
# Cell 7 - Reusable pipeline utilities. All paths in manifests are repository-relative.
import copy
PIPE_DIR = BASE_DIR / 'data/pipeline_v1/documents'
PIPELINE_RUNS = {}
CONFIG_C = {
    'pipeline_version': 'plantpal-week3-v1',
    'admitted_pages': {'S01':[5,10], 'S03':[6]},
    'method': 'paragraph-aware', 'target_characters':500, 'overlap':0,
    'page_local':True, 'section_rules_version':'verified-nparks-2026-11-10-v1',
    'parser_version':'6.10.0',
}
VERIFIED_SECTIONS = {
    ('S01', 5): ['Indoor Garden\nConsiderations', 'Space', 'Light', 'Water', 'Plant Choice'],
    ('S01', 10): ['Plant Necessity -\nMedia', 'Charcoal', 'Coco Chips'],
}

def inspected_inputs(base, sources, cfg):
    if cfg['method'] != 'paragraph-aware' or cfg['overlap'] != 0 or not cfg['page_local']:
        raise ValueError('Unsupported V1 configuration; review Part B before changing method.')
    if cfg['target_characters'] <= 0:
        raise ValueError('Character target must be positive.')
    if cfg['section_rules_version'] != 'verified-nparks-2026-11-10-v1':
        raise ValueError('Unknown section rule version.')
    if importlib.metadata.version('pypdf') != cfg['parser_version']:
        raise ValueError('Parser version differs from verified configuration.')
    if len({s['source_id'] for s in sources}) != len(sources):
        raise ValueError('Duplicate source IDs.')
    raw_root = (base/'data/raw').resolve()
    texts, inspection, fingerprints = {}, [], []
    for sid, selected in cfg['admitted_pages'].items():
        s = next(s for s in sources if s['source_id']==sid)
        path = (raw_root/s['filename']).resolve()
        path.relative_to(raw_root)
        if not path.is_file():
            raise FileNotFoundError(f'Missing input: {s["filename"]}')
        digest = sha256_file(path)
        if digest != s['expected_sha256']:
            raise ValueError(f'Source checksum changed: {sid}; review before admission.')
        reader = PdfReader(path)
        if reader.is_encrypted:
            raise ValueError(f'Encrypted input: {sid}')
        texts[sid] = [page.extract_text() or '' for page in reader.pages]
        if not selected or len(set(selected)) != len(selected):
            raise ValueError('Select unique page numbers.')
        for page in selected:
            if not 1 <= page <= len(reader.pages) or not texts[sid][page-1].strip():
                raise ValueError(f'Unreadable selected page: {sid}/{page}')
        fingerprints.append({'source_id':sid,'path':path.relative_to(base).as_posix(),
                             'sha256':digest,'url':s['url'],'title':s['title']})
        inspection.append({'source_id':sid,'total_pages':len(reader.pages),'admitted_pages':selected,
                           'selected_raw_characters':[len(texts[sid][n-1]) for n in selected]})
    return texts, inspection, fingerprints

def clean_selected_pages(page_text, cfg):
    PAGE_DOCS = []
    for sid, pages in cfg['admitted_pages'].items():
        for page in pages:
            raw = page_text[sid][page-1]
            if (sid, page) == ('S01', 10):
                noise = 'Can insert any relevant image here?'
                assert raw.count(noise) == 1
                raw = raw.replace(noise, '')
            if sid == 'S01':
                markers = VERIFIED_SECTIONS[(sid, page)]
                # Match complete heading lines, not words inside the prose.
                positions = []
                for marker in markers:
                    matches = list(re.finditer(r'(?m)^'+re.escape(marker)+r'\s*$', raw))
                    assert len(matches) == 1, (sid, page, marker)
                    positions.append(matches[0].start())
                assert positions == sorted(positions) and positions[0] == 0
                blocks = [raw[a:b] for a,b in zip(positions, positions[1:]+[len(raw)])]
                # Keep page heading with first content section, not as an orphan chunk.
                if page == 5:
                    blocks = [blocks[0]+blocks[1]] + blocks[2:]
            else:
                assert raw.rstrip().endswith('Soil Mix')
                raw = raw.rstrip()[:-len('Soil Mix')].rstrip()
                starts = [m.start() for m in re.finditer(r'(?m)^(?:Topsoil refers|Compost refers|Different plants|Sand refers)', raw)]
                assert len(starts) == 4 and starts[0] == 0
                blocks = [raw[a:b] for a,b in zip(starts, starts[1:]+[len(raw)])]
                blocks[0] = 'Soil Mix\n'+blocks[0]
            blocks = [re.sub(r'\s+', ' ', b).strip() for b in blocks]
            assert all(blocks)
            PAGE_DOCS.append({'source_id': sid, 'page': page, 'blocks': blocks,
                              'text': '\n\n'.join(blocks)})
    return PAGE_DOCS

def pack_sections(text, target):
    spans, start, end = [], None, None
    for block in re.finditer(r'[^\n]+', text):
        if start is not None and block.end()-start > target:
            spans.append((start,end))
            start = None
        if start is None:
            start = block.start()
        end = block.end()
    if start is not None:
        spans.append((start,end))
    return spans

def prepare_records(page_docs, sources, cfg):
    registry = {s['source_id']:s for s in sources}
    output = []
    for d in page_docs:
        s = registry[d['source_id']]
        spans = pack_sections(d['text'],cfg['target_characters'])
        if re.sub(r'\s+','', ''.join(d['text'][a:b] for a,b in spans)) != re.sub(r'\s+','',d['text']):
            raise ValueError('Chunking lost non-whitespace content.')
        for a,b in spans:
            i = len(output)+1
            chunk = {'source_id':d['source_id'],'page':d['page'],'start':a,'end':b,'text':d['text'][a:b].strip()}
            output.append(dict(chunk,record_id=f"{d['source_id']}-p{d['page']:02d}-{i:04d}",
                source=s['filename'],title=s['title'],url=s['url'],source_sha256=s['expected_sha256'],
                processing=f"verified-sections; paragraph-aware; target-char-{cfg['target_characters']}; overlap-0; page-local",
                coverage_note='S03 p6: prose only; extracted text omits soil-ratio table.' if s['source_id']=='S03' else 'Selected page text only.'))
    return output

def validate_records(exported, expected, page_docs):
    if not exported or exported != expected:
        raise ValueError('Prepared JSONL is empty or differs from expected records.')
    if len({r['record_id'] for r in exported}) != len(exported):
        raise ValueError('Duplicate record IDs.')
    lookup = {(d['source_id'],d['page']):d['text'] for d in page_docs}
    for r in exported:
        text = lookup[(r['source_id'],r['page'])]
        if not r['text'].strip() or r['text'] != text[r['start']:r['end']].strip():
            raise ValueError('Invalid prepared text or page span.')
        if not r['title'] or not r['url'].startswith('https://'):
            raise ValueError('Missing citation metadata.')

def execute_document_pipeline(base, sources, configuration):
    base = Path(base).resolve()
    cfg = copy.deepcopy(configuration)
    page_text, inspection, fingerprints = inspected_inputs(base,sources,cfg)
    page_docs = clean_selected_pages(page_text,cfg)
    prepared = prepare_records(page_docs,sources,cfg)
    out = base/'data/pipeline_v1/documents'
    out.mkdir(parents=True,exist_ok=True)
    target = out/'prepared_text.jsonl'
    # Validate a temporary export before replacing the last valid output.
    staging = out/'prepared_text.jsonl.tmp'
    try:
        staging.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in prepared),encoding='utf-8')
        loaded = [json.loads(line) for line in staging.read_text(encoding='utf-8').splitlines()]
        validate_records(loaded,prepared,page_docs)
        if not staging.stat().st_size:
            raise ValueError('Empty prepared file.')
        staging.replace(target)
    finally:
        staging.unlink(missing_ok=True)
    mappings = [{'source_id':s['source_id'],'source':s['path'],
                 'output':target.relative_to(base).as_posix(),
                 'pages':cfg['admitted_pages'][s['source_id']],
                 'records':sum(r['source_id']==s['source_id'] for r in prepared)} for s in fingerprints]
    record = {'pipeline_version':cfg['pipeline_version'],
              'generated_at_utc':datetime.now(timezone.utc).isoformat(),
              'execution_environment':'Google Colab' if IN_COLAB else 'local Python',
              'python_version':platform.python_version(),'configuration':cfg,'data_type':'documents',
              'sources':fingerprints,'inspection':inspection,'source_output_mapping':mappings,
              'validation':{'output_count':1,'record_count':len(prepared),'output_bytes':target.stat().st_size,
                            'output_sha256':sha256_file(target),'record_checks':'PASS'},
              'outputs':[target.relative_to(base).as_posix()],
              'limitations':['Selected prose pilot, not a complete answer corpus','Manual source-specific section rules',
                            'S02 deferred','S03 p6 table not extracted','No retrieval or model evaluation'],
              'access_rights':'Personal working copies only; no open redistribution licence established. Prepared text stays local.',
              'source_version':'Pinned by source SHA-256; publication dates not verified.'}
    manifest = out/'pipeline_manifest.json'
    manifest.write_text(json.dumps(record,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    saved = json.loads(manifest.read_text(encoding='utf-8'))
    if saved != record or sha256_file(target) != saved['validation']['output_sha256']:
        raise ValueError('Pipeline manifest verification failed.')
    print('SUCCESS: documents | inputs:',len(fingerprints),'| outputs: 1 | records:',len(prepared))
    print('Manifest:',manifest.relative_to(base).as_posix())
    return record

print('Cell 7 ready: document pipeline utilities defined.')


Cell 7 ready: document pipeline utilities defined.


In [14]:
# Cell 8A - Run the selected document pathway using Part B's configuration.
first_run = execute_document_pipeline(BASE_DIR,SOURCES,CONFIG_C)
PIPELINE_RUNS['documents'] = {'first':first_run,
    'runner':lambda: execute_document_pipeline(BASE_DIR,SOURCES,CONFIG_C)}
# This comparison reads Part B only AFTER independent preparation from raw PDFs.
assert sha256_file(PIPE_DIR/'prepared_text.jsonl') == sha256_file(PROCESSED_DIR/'prepared_text.jsonl'), 'Unexpected difference from Part B'
print('Part B / Part C prepared JSONL byte-for-byte comparison: PASS')


SUCCESS: documents | inputs: 2 | outputs: 1 | records: 5
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Part B / Part C prepared JSONL byte-for-byte comparison: PASS


In [15]:
# Cell 9 - Repeatability and provenance verification; timestamps excluded deliberately.
assert PIPELINE_RUNS
repeatability = []
for kind, entry in PIPELINE_RUNS.items():
    first, second = entry['first'], entry['runner']()
    for field in ['sources','validation','source_output_mapping','configuration']:
        assert first[field] == second[field], f'Repeatability mismatch: {field}'
    repeatability.append({'pathway':kind,'status':'PASS','source_count':len(second['sources']),
        'record_count':second['validation']['record_count'],
        'prepared_sha256':second['validation']['output_sha256'],
        'matching_fields':['sources','validation','source_output_mapping','configuration'],
        'timestamp_comparison':'Excluded: run timestamps may differ'})
    print('PASS:',kind,'- source/output fingerprints, sizes, counts and mappings match.')

# Exercise two actual failure responses without modifying any original source.
failure_checks = []
for label in ['missing_input','changed_source_checksum']:
    altered = copy.deepcopy(SOURCES)
    if label == 'missing_input':
        altered[0]['filename'] = '__plantpal_missing_input_test__.pdf'
        assert not (RAW_DIR/altered[0]['filename']).exists()
        expected_exception = FileNotFoundError
    else:
        altered[0]['expected_sha256'] = '0'*64
        expected_exception = ValueError
    before = sha256_file(PIPE_DIR/'prepared_text.jsonl')
    try:
        execute_document_pipeline(BASE_DIR,altered,CONFIG_C)
    except expected_exception:
        assert before == sha256_file(PIPE_DIR/'prepared_text.jsonl')
        failure_checks.append({'case':label,'result':'PASS: stopped before output replacement'})
    else:
        raise AssertionError(f'Failure case was not rejected: {label}')
print('Failure-response checks:',failure_checks)
evidence = {'stage':'week3-part-c','repeatability':repeatability,'failure_checks':failure_checks,
    'part_b_output_match':True,'execution_environment':'Google Colab' if IN_COLAB else 'local Python',
    'limitation':'Local repeatability smoke check, not cross-environment reproducibility or content correctness.'}
evidence_path = BASE_DIR/'data/inspection/week3_part_c_validation.json'
evidence_path.write_text(json.dumps(evidence,indent=2)+'\n',encoding='utf-8')
print('All selected pipelines passed repeatability smoke checks.')


SUCCESS: documents | inputs: 2 | outputs: 1 | records: 5
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents - source/output fingerprints, sizes, counts and mappings match.
Failure-response checks: [{'case': 'missing_input', 'result': 'PASS: stopped before output replacement'}, {'case': 'changed_source_checksum', 'result': 'PASS: stopped before output replacement'}]
All selected pipelines passed repeatability smoke checks.


### C2-C4 - Documentation and repository handoff

- Workflow: `docs/week3_pipeline_diagram.png` and editable `.drawio` source. Expands A5 data preparation for A4.
- Decisions and stage contracts: `docs/week3_pipeline_decisions.md`.
- Execution manifest: `data/pipeline_v1/documents/pipeline_manifest.json`.
- Repeatability/failure evidence: `data/inspection/week3_part_c_validation.json`.
- Full prepared JSONL remains local and ignored. Public metadata references local output regenerated by Run All.
- Actual local sample review confirms the same five chunks reviewed in Part B; byte-for-byte equality is checked.
- All A/B/C cells were executed locally. No Colab runtime, retrieval quality, model answer correctness,
  open redistribution permission, or complete source coverage is claimed.

Reference: Topic 3 Lab, C1-C4 and submission notes, pp. 41-65.
